In this notebook, I am going to:
 1. Understand the structure of the dataset. 
 2. Verify the geometric quality(Are the embeddings well distributed on the 64 dimention sphere)
 3. Verify the semantic discriminativeness

In [5]:
import ee # Earth Engine Python API 
import geemap
from IPython.display import display, HTML

In [6]:
# Authenticate and initialize the Earth Engine library.
ee.Authenticate()
ee.Initialize(project="iconic-smoke-480408-r8")

In [7]:
# Initialize the dataset
dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

In [8]:
type(dataset)

ee.imagecollection.ImageCollection

In [9]:
help(dataset)
# https://developers.google.com/earth-engine/apidocs/ee-imagecollection

Help on ImageCollection in module ee.imagecollection object:

class ImageCollection(ee.collection.Collection)
 |  ImageCollection(*args, **kwargs)
 |
 |  Representation for an Earth Engine ImageCollection.
 |
 |  Method resolution order:
 |      ImageCollection
 |      ee.collection.Collection
 |      typing.Generic
 |      ee.element.Element
 |      ee.computedobject.ComputedObject
 |      ee.encodable.Encodable
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  And(self) -> image.Image
 |      Returns an image with the logical AND of the collection.
 |
 |      Reduces an image collection by setting each pixel to 1 if and only if all
 |      the non-masked values at that pixel are non-zero across the stack of all
 |      matching bands. Bands are matched by name.
 |
 |  Or(self) -> image.Image
 |      Returns an image with the logical OR of the collection.
 |
 |      Reduces an image collection by setting each pixel to 1 if and only if any of
 |      the non-masked values at 

In [10]:
dataset.elementType()

ee.image.Image

In [11]:
help(dataset.elementType())
# https://developers.google.com/earth-engine/apidocs/ee-image

Help on class Image in module ee.image:

class Image(ee.element.Element)
 |  Image(*args, **kwargs)
 |
 |  An object to represent an Earth Engine image.
 |
 |  Method resolution order:
 |      Image
 |      ee.element.Element
 |      ee.computedobject.ComputedObject
 |      ee.encodable.Encodable
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  And(self, image2: _arg_types.Image) -> Image
 |      Returns 1 if both values are non-zero; otherwise 0.
 |
 |      Returns 1 if and only if both values are non-zero for each matched pair of
 |      bands in image1 and image2.
 |
 |      If either image1 or image2 has only 1 band, then it is used against all the
 |      bands in the other image. If the images have the same number of bands, but
 |      not the same names, they're used pairwise in the natural order. The output
 |      bands are named for the longer of the two inputs, or if they're equal in
 |      length, in image1's order. The type of the output pixels is boolean.
 |
 

In [12]:
firstImage = dataset.first()

# Print the first image's metadata
imageInfo = firstImage.getInfo()

import json
with open("info.json", "w") as f:
    json.dump(imageInfo, f, indent=2)

print("Saved to info.json")

Saved to info.json


In [13]:
imageInfo["bands"][0]

{'id': 'A00',
 'data_type': {'type': 'PixelType', 'precision': 'double'},
 'dimensions': [16384, 16384],
 'crs': 'EPSG:32724',
 'crs_transform': [10, 0, 827680, 0, 10, 9344640]}

"crs": "EPSG:32724"

Meaning
	•	Coordinate Reference System = UTM Zone 24S
	•	EPSG:32724 → UTM, Southern Hemisphere, Zone 24

In [14]:
# Open Map
m = geemap.Map()
lon = 121.5036
lat = 31.2
m.set_center(lon, lat, zoom=12)
display(m)

Map(center=[31.2, 121.5036], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [15]:
point = ee.Geometry.Point(lon, lat)

img = dataset.filterDate("2022-01-01", "2023-01-01").filterBounds(point).first()

img

In [16]:
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}
m.add_ee_layer(img, vis_params, name="embeddings")

m

Map(center=[31.2, 121.5036], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [17]:
roi = ee.Geometry.Rectangle([120.8, 30.7, 122.2, 31.9])

# Get the 2024 embedding image covering around Ningbo
img_2024 = (
    dataset
    .filterBounds(roi)
    .filterDate("2024-01-01", "2024-12-31")
    .first()
)

In [18]:
help(ee.Image.sample)

Help on function sample in module ee.image:

sample(
    self,
    region: _arg_types.Geometry | None = None,
    scale: _arg_types.Number | None = None,
    projection: _arg_types.Projection | None = None,
    factor: _arg_types.Number | None = None,
    numPixels: _arg_types.Integer | None = None,
    seed: _arg_types.Integer | None = None,
    dropNulls: _arg_types.Bool | None = None,
    tileScale: _arg_types.Number | None = None,
    geometries: _arg_types.Bool | None = None
) -> featurecollection.FeatureCollection
    Returns an ee.FeatureCollection of samples from an ee.Image.

    Samples the pixels of an image, returning them as a FeatureCollection. Each
    feature will have 1 property per band in the input image. Note that the
    default behavior is to drop features that intersect masked pixels, which
    result in null-valued properties (see dropNulls argument).

    Args:
      region: The region to sample from. If unspecified, uses the image's whole
        footprint.
  

In [19]:
# https://developers.google.com/earth-engine/apidocs/ee-image-sample
samples = img_2024.sample(
    region=roi,
    scale=10,
    numPixels=5000,
    geometries=True  # keep geometry to map clusters later
)

In [20]:
k = 5
clusterer = ee.Clusterer.wekaKMeans(nClusters=k).train(
    features=samples,
    inputProperties=img_2024.bandNames()
)

result = img_2024.cluster(clusterer)

In [21]:
m = geemap.Map(center=[31.2304, 121.4737], zoom=9)

palette = ['#ff0000', '#00ff00', '#0000ff', '#ffff00', '#800080']  # length = k
vis_params = {'min': 0, 'max': k - 1, 'palette': palette}

m.add_ee_layer(result.visualize(**vis_params), {}, "KMeans Clusters 2024 (Shanghai)")
m.add_layer_control()
display(m)

# --- 7) Legend ---
legend_items = "".join(
    f"""
    <div style="display:flex;align-items:center;margin:4px 0;">
      <div style="width:20px;height:14px;background:{color};margin-right:8px;border:1px solid #444;"></div>
      <div style="font-size:13px;color:#222;">Cluster {i}</div>
    </div>
    """
    for i, color in enumerate(palette)
)

legend_html = f"""
<div style="font-family: Arial, Helvetica, sans-serif;
            background: white;
            padding:10px;
            border-radius:6px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.3);
            display:inline-block;">
  <div style="font-weight:600;margin-bottom:6px;">Cluster ID</div>
  {legend_items}
</div>
"""

display(HTML(legend_html))

Map(center=[31.2304, 121.4737], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…